# Hangman — round 3 on a wider model

Continues from the finished 3-round run instead of repeating it. Loads the
saved 6.96M-state buffer and the round-2 checkpoint, collects one more
on-policy round with that model, and trains a **wider** network from scratch
on the aggregated ~9.2M states.

Two reasons to expect a gain:

- Round 1 → 2 was still worth +2.6 points on the holdout, so the aggregation
  had not plateaued.
- The round-2 model saw 7M states with only 8 layers at `d_model=256`, and its
  training loss was still drifting down at the end of epoch 3. The states are
  outgrowing the network, not the other way round.

**Inputs to attach:** the competition, `hangman-src`, and `hangman-weights` —
and `hangman-weights` must contain `states_round2.npz` as well as
`hangman_r2.pt`. If you created that dataset without the state buffer, add it
from the training run's Output tab first, or this notebook has to recollect
all three rounds from scratch.

| checkpoint | holdout | test.txt |
|---|---|---|
| n-gram baseline | 48.50% | 50.80% |
| round 0 | 50.22% | — |
| round 1 | 63.32% | — |
| round 2 | 65.96% | **67.78%** |
| round 3 | ? | ? |


In [ ]:
import os, sys, time, json, glob, warnings
warnings.filterwarnings("ignore")

# Match whatever your dataset path actually is (check with !ls /kaggle/input).
SRC = "/kaggle/input/datasets/suniljadaun/hangman-src/src"
WEIGHTS = "/kaggle/input/datasets/suniljadaun/hangman-weights"
sys.path.insert(0, SRC)
sys.path.insert(0, "/kaggle/working/src")

import numpy as np
import torch

from hangman.data import load_words, overlap, split_holdout, find_competition_dir
from hangman.policies import LengthFrequencyPolicy, NeuralPolicy, EpsilonMixPolicy
from hangman.policies.ngram import NGramPolicy
from hangman.states import collect, BoardBatcher, StateBuffer
from hangman.model import HangmanNet, ModelConfig
from hangman.train import train, TrainConfig
from hangman.evaluate import evaluate, print_report

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
print("weights dir:", sorted(os.listdir(WEIGHTS)))

In [ ]:
COMP = find_competition_dir()
train_words_all = load_words(f"{COMP}/train.txt")
test_words = load_words(f"{COMP}/test.txt")
assert overlap(train_words_all, test_words)["test_words_in_train"] == 0

MAX_LEN = max(max(map(len, train_words_all)), max(map(len, test_words)))

# Same seed as the original run, so the holdout is the same 10k words and the
# round-by-round numbers stay comparable.
TRAIN_WORDS, HOLDOUT = split_holdout(train_words_all, n_holdout=10_000, seed=0)
print(len(TRAIN_WORDS), len(HOLDOUT), "MAX_LEN =", MAX_LEN)

In [ ]:
ckpt = torch.load(f"{WEIGHTS}/hangman_r2.pt", map_location=DEVICE)
r2 = HangmanNet(ModelConfig(**ckpt["cfg"]))
r2.load_state_dict(ckpt["model"])
r2_policy = NeuralPolicy(r2, DEVICE, fusion=0.3)

buffer = StateBuffer.load(f"{WEIGHTS}/states_round2.npz")
print(f"loaded {len(buffer):,} states from the previous run")

## Round 3 state collection

Explore against the n-gram heuristic exactly as before, so the state
distribution stays comparable across rounds.

In [ ]:
heuristic = NGramPolicy(TRAIN_WORDS)
explore = EpsilonMixPolicy(r2_policy, heuristic, epsilon=0.02, noise=0.05, seed=200)

t = time.time()
new_states, stats = collect(TRAIN_WORDS, explore, max_len=MAX_LEN)
print(f"collected {len(new_states):,} in {time.time()-t:.0f}s; generator {stats}")

buffer = StateBuffer.concat([buffer, new_states])
buffer.save("/kaggle/working/states_round3.npz")
print(f"aggregated buffer: {len(buffer):,} states")

## Train a wider model from scratch on everything

`d_model` 256 → 384 and 8 → 10 layers, roughly 2.7× the parameters. Retraining
from scratch rather than fine-tuning keeps the round-2 policy's blind spots out
of the new model, which is the same reason every earlier round restarted.

Budget roughly 2–3 hours on a T4. **Run this as Save & Run All (Commit)**, not
interactively.

In [ ]:
MODEL_CFG = ModelConfig(d_model=384, n_layers=10, n_heads=8, d_ff=1536,
                        dropout=0.1, max_len=MAX_LEN)
batcher = BoardBatcher(TRAIN_WORDS, buffer, max_len=MAX_LEN,
                       batch_size=512, bucket=True, seed=3)
cfg = TrainConfig(epochs=3, batch_size=512, lr=3e-4,
                  amp=(DEVICE == "cuda"), model=MODEL_CFG, seed=3)
print(f"{len(buffer):,} states, {len(batcher):,} batches/epoch")

model = train(batcher, cfg, device=DEVICE)
torch.save({"model": model.state_dict(), "cfg": MODEL_CFG.__dict__},
           "/kaggle/working/hangman_r3.pt")

## Evaluate, and tune fusion on the *whole* holdout

The previous sweep used 4,000 words, where a 0.8-point standard error made
every fusion value statistically indistinguishable and the "best" pick was
noise. Using all 10,000 halves that, and ties on win rate are broken by mean
strikes — which is exactly how the competition breaks ties.

In [ ]:
results = []
for fusion in [0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0]:
    m = evaluate(HOLDOUT, NeuralPolicy(model, DEVICE, fusion=fusion), max_len=MAX_LEN)
    results.append((m["win_rate"], -m["mean_wrong"], fusion))
    print(f"fusion={fusion:.1f}  win {m['win_rate']:.2f}%  strikes {m['mean_wrong']:.3f}")

best = max(results)
print(f"\nbest fusion {best[2]} -> {best[0]:.2f}%  (round 2 scored 65.96%)")
json.dump({"fusion": best[2]}, open("/kaggle/working/inference.json", "w"))
print_report(evaluate(HOLDOUT, NeuralPolicy(model, DEVICE, fusion=best[2]), max_len=MAX_LEN))

## Score it on test.txt directly

No need for a separate submission notebook this time — write the file here and
compare against the 67.776% already banked. **Submit only if this is higher.**

In [ ]:
from hangman.submit import write_submission, validate_submission

metrics = evaluate(test_words, NeuralPolicy(model, DEVICE, fusion=best[2]),
                   max_len=MAX_LEN)
print_report(metrics)
print(f"\nround 2 was 67.776% / 3.627 strikes")
print(f"round 3 is  {metrics['win_rate']:.3f}% / {metrics['mean_wrong']:.3f} strikes")

write_submission(metrics["guesses"], "submission.csv")
validate_submission("submission.csv", expected_rows=len(test_words))